# Hardware-informed versus uniform-noise stabilizer-code search

**Numerical comparison — 13 November 2026.** All results are intermediate, based on a reduced neutral-atom Rydberg noise model and a four-qubit code-capacity experiment.

**Research question:** does a SAT-designed code chosen using the experimentally motivated Rydberg Pauli distribution outperform a code chosen assuming uniformly likely Pauli errors and **no Hamiltonian seed**?

The comparison holds fixed the physical noise on which codes are tested: **two independent identical Rydberg-pair channels** on qubits (1,2) and (3,4). Each code encodes one logical qubit into four physical qubits, has distance at least 2, and is decoded with exact maximum-likelihood Pauli decoding.

This is **not** a measured four-atom channel; cross-pair correlations, leakage correction, actual syndrome circuits, and coherent non-Pauli effects are excluded. The SAT method enforces code constraints; exhaustive evaluation chooses the best code by probability (not a weighted MaxSAT optimization).

## 1. Matched noise models

Let (q_P) be the extracted two-atom computational Pauli weights. The *design-only* uniform model keeps (q_{II}) unchanged but divides the remaining probability evenly between all fifteen nonidentity two-qubit Paulis:

[
q^{\mathrm{uniform}}_{II}=q_{II},\qquad
q^{\mathrm{uniform}}_{P\ne II}=\frac{\sum_{Q\ne II}q_Q}{15}.
]

Thus both models have identical total two-atom Pauli error probability and computational survival; they differ **only in error type**. All final evaluations use the **Rydberg** model, not the model used for selection.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rydberg_sat_study import (pair_probabilities, enumerate_stabilizers, pauli_to_bits, bits_to_pauli,
                               span, distance, ml_success, correctable_error_set)

q, w_hw = pair_probabilities("rydberg_pair_pauli_weights.json")
survival = sum(q.values())
pauli_error = survival - q["II"]
q_uniform = {P: (q["II"] if P == "II" else pauli_error/15) for P in q}

def paired_weights(dist):
    out = np.zeros(256)
    for P,a in dist.items():
        for Q,b in dist.items():
            out[pauli_to_bits(P + Q)] = a*b
    return out

w_uniform = paired_weights(q_uniform)
assert np.isclose(w_uniform.sum(), w_hw.sum())
print(f"Two-pair computational survival: {100*w_hw.sum():.6f}%")
print(f"Conditional nonidentity per atom pair: {100*pauli_error/survival:.6f}%")

## 2. SAT constraints and exact decoding

We use arbitrary Pauli generators with three independent mutually commuting stabilizers, corresponding to ([[4,1]]). Every admitted code has distance (d\ge2). We impose **no extra requirement to correct only (Z) errors**, because that would already tell the uniform-noise designer which errors are important.

For a stabilizer group (S), our exact Pauli decoder considers all (256) errors, including degenerate errors, and selects the highest-probability stabilizer-equivalent error class for each syndrome. The conditional logical failure is

[
p_{L\mid\mathrm{survive}}=\frac{p_{\mathrm{survive}}-p_{\mathrm{ML-success}}}{p_{\mathrm{survive}}}.
]

The previous SAT/SMT backend verifies code feasibility. Here, **exhaustive enumeration of all 11,475 rank-3 groups** and all 2,268 distance-2 groups certifies the finite-search optima.

In [ ]:
all_groups = enumerate_stabilizers()
d2 = [(G,b) for G,b in all_groups.items() if distance(G,b) >= 2]
assert len(all_groups) == 11475 and len(d2) == 2268

def failure(G,b,weights):
    success,_ = ml_success(G,b,weights)
    return (weights.sum()-success)/weights.sum()

hw_scores = np.array([failure(G,b,w_hw) for G,b in d2])
uniform_scores = np.array([failure(G,b,w_uniform) for G,b in d2])
best_hw = hw_scores.min()
best_uniform = uniform_scores.min()
uniform_ties = np.flatnonzero(np.isclose(uniform_scores,best_uniform,rtol=0,atol=1e-12))
assert len(uniform_ties) == 324
print("Hardware-best conditional failure:",100*best_hw,"%")
print("Number of uniform-optimal codes:",len(uniform_ties))
print("Range under actual Rydberg noise:",100*hw_scores[uniform_ties].min(),
      "to",100*hw_scores[uniform_ties].max(),"%")

## 3. Code generators and primary comparison

Every code is evaluated **under the actual Rydberg Pauli distribution**, irrespective of design model.

| Design assumption | Stabilizer generators | Conditional logical failure |
|:--|:--|--:|
| Rydberg probabilities, no H seed | `XXXX, ZYII, XXYZ` | **0.03786664%** |
| Rydberg probabilities, require (ZZII) seed | `XXXX, ZZII, XXYZ` | 0.03786803% |
| Uniform two-qubit Pauli noise, no H seed | `XXII, IIXX, ZZZZ` | 0.11681243% |

The uniform code is **one** lexically chosen optimum under uniform noise. The hardware-informed code reduces failure by **67.6%** relative to that example. The selection of a different tied uniform optimum can substantially change this result.

In [ ]:
candidate_codes = {
    "Hardware-informed": ["XXXX","ZYII","XXYZ"],
    "Hardware + H seed": ["XXXX","ZZII","XXYZ"],
    "Uniform-noise CSS": ["XXII","IIXX","ZZZZ"],
}
values = {}
for name, labels in candidate_codes.items():
    basis = list(map(pauli_to_bits,labels))
    G=span(basis)
    assert len(G)==8 and distance(G,basis)>=2
    values[name]=failure(G,basis,w_hw)
    print(f"{name:21s}: {100*values[name]:.8f}%")

basis=list(map(pauli_to_bits,candidate_codes["Uniform-noise CSS"]))
G=span(basis)
assert abs(failure(G,basis,w_uniform)-best_uniform)<1e-12

plt.figure(figsize=(9,4))
plt.bar(list(values), [100*v for v in values.values()])
plt.ylabel("Conditional logical error under Rydberg noise (%)")
plt.title("Same Rydberg evaluation channel")
plt.xticks(rotation=9)
plt.show()

## 4. The important tie: what hardware information really buys us

Under uniform two-qubit Pauli noise, **324 codes are equally optimal**. When tested under Rydberg noise their conditional logical failures range from **0.03786664%** to **0.11681243%**. The median is **0.03859042%** and the mean is **0.05202529%**.

**64 of these 324 tied uniform-model optima achieve the same actual Rydberg performance as the informed optimum** (within numerical precision). So we **cannot** claim the hardware-informed code outperforms *every* code that could have been found under uniform noise.

The supported finding is narrower and still useful: **hardware probabilities distinguish among codes that uniform noise cannot rank.** The improvement over the **median** uniform optimum is approximately **1.88%**. The improvement over the particular conventional CSS representative above is **67.6%**.

In [ ]:
tie_actual = 100*hw_scores[uniform_ties]
print("Uniform-optimal count:",len(tie_actual))
print("Under Rydberg noise: min / median / mean / max (%):",
      np.min(tie_actual),np.median(tie_actual),np.mean(tie_actual),np.max(tie_actual))
print("Number as good as hardware optimum:",
      np.sum(abs(hw_scores[uniform_ties]-best_hw)<1e-12))
plt.figure(figsize=(9,4))
plt.hist(tie_actual,bins=22)
plt.axvline(100*best_hw,ls="--",label="Hardware-informed optimum")
plt.axvline(np.median(tie_actual),ls=":",label="Uniform-optimal median")
plt.xlabel("Logical failure under Rydberg noise (%)")
plt.ylabel("Uniform-optimal code count")
plt.legend()
plt.show()

## 5. Loss and residual Rydberg population

For the assumed two independent pairs, the probability that **both pairs finish in the computational subspace** is approximately **99.693208%**. The **remaining 0.306792%** is not handled by our Pauli-only decoder.

If *every* such outcome is pessimistically counted as an unsuccessful correction, the corresponding failure **proxy** is:

| Design | Failure proxy, including all out-of-computational outcomes |
|:--|--:|
| Informed | **0.344542%** |
| Uniform CSS representative | **0.423246%** |

The relative change becomes about **18.6%**. This is **not** a calibrated logical error rate for a loss-aware decoder, and not every physical loss necessarily means logical failure.

In [ ]:
surv4 = w_hw.sum()
for name,value in values.items():
    proxy = (1-surv4) + surv4*value
    print(f"{name:21s}: {100*proxy:.6f}% (loss-counted-as-failure proxy)")
print("Out-of-computational rate:",100*(1-surv4),"%")

## 6. Conclusions and controls needed

**Supported:** An exact finite-size search shows that probability information can break degeneracies left by a uniform two-qubit error model. With the current Rydberg data, choosing the best code improves over many—but not all—uniform-noise choices. Adding the H seed (ZZII) offers no meaningful logical-error improvement in this example.

**Not yet established:** an unconditional hardware-informed advantage, a measured physical error-correction experiment, a pseudothreshold, a leakage-correcting code, or scalable weighted MaxSAT. The uniform 15-Pauli model is one particular hardware-unaware baseline; independent one-qubit depolarizing noise is another and yields even more ties.

**Next steps:** evaluate under independent or perturbed physical noise, implement loss/erasure treatment, include circuit-level syndrome extraction, test larger codes and an actual weighted SAT objective. See the earlier Rydberg physics and SAT results notebooks for input provenance.